# A1 Embedding Study — Phase 1

Thin report notebook: all computation lives in `../src/`. This notebook loads archived run artifacts and states answers required by the plan.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

RESEARCH = Path('..').resolve()
sys.path.insert(0, str(RESEARCH))

from src.reporting import discover_runs, load_json, latest_run_dir

runs = discover_runs()
runs

## Data identity

Corpus: `stanfordnlp/imdb` train[11500:13500], 1000 neg + 1000 pos, 470,407 tokens after assignment preprocessing.

In [ ]:
stats = load_json(Path(runs['module_a']) / 'corpus_stats.json')
print('neg:', stats['neg_labels'], 'pos:', stats['pos_labels'])
print('tokens:', stats['token_count'])
print('dataset revision:', stats['dataset_revision'])
print('raw sha256:', stats['raw_text_sha256'])
print('token sha256:', stats['preprocessed_token_sha256'])
print('eval vocab (after excluding "", START, END):', stats['eval_vocab_size'])
print('count>=10 before exclusion:', stats['count_ge_10_before_exclusion'])

## Module A — Toy corpus + neighbor comparison

Hand-checkable PMI/PPMI on a toy corpus, then IMDB subset at k=50.

In [ ]:
toy = load_json(Path(runs['module_a']) / 'toy_corpus_check.json')
pd.DataFrame(toy['examples'])

In [ ]:
neighbors = load_json(Path(runs['module_a']) / 'module_a_neighbors.json')
overlap = load_json(Path(runs['module_a']) / 'module_a_overlap.json')

for pipeline in ['raw_svd', 'ppmi_svd', 'ppmi_cds_svd']:
    print('\n===', pipeline, 'seed=0 top-10 ===')
    for q, res in neighbors[pipeline]['seed_0']['queries'].items():
        if res.get('evaluable'):
            words = [w for w, s in res['neighbors']]
            print(f"  {q}: {words}")
        else:
            print(f"  {q}: UNEVALUABLE ({res.get('reason')})")

In [ ]:
print('Seed overlap vs seed 0 (top-10):')
for pipeline, seeds in overlap.items():
    print(pipeline, seeds)

### Module A answers

- **High count ≠ strong association:** raw co-occurrence favors function words and any frequent pair (e.g. *the*–*a* in the toy corpus) regardless of semantic relation.
- **PMI amplifies rare events:** once-only pairs get high PMI because joint probability is compared to independent marginals; rare co-occurrences dominate.
- **PPMI vs raw neighbors differ:** PPMI down-weights ubiquitous pairs and emphasizes informative co-occurrences, so neighbors shift from frequency-driven tokens toward contextual associates (compare *raw_svd* vs *ppmi_svd* tables above).

## Module B — Dimension, reconstruction, PIP, rotation

Single PPMI fit at k=100; lower k uses column prefixes of the same `fit_transform` output.

In [ ]:
metrics = load_json(Path(runs['module_b']) / 'module_b_metrics.json')
print('Reconstruction squared error ||M||_F^2 - ||E_k||_F^2:')
for k, v in metrics['reconstruction_squared_error'].items():
    print(f'  k={k}: {v:.2f}')
print('\nPIP loss (subset S, ref k=100):')
for k, v in metrics['pip_loss'].items():
    print(f'  k={k}: {v:.4f}')

In [ ]:
distortion = load_json(Path(runs['module_b']) / 'module_b_2d_distortion.json')
rotation = load_json(Path(runs['module_b']) / 'module_b_rotation_check.json')
print('2D vs k=100 neighbor Jaccard (sample):')
for q, d in distortion.items():
    print(f"  {q}: {d['jaccard']:.3f}")
print('\nRotation check:', rotation['pip_unchanged'], 'tolerance', rotation['tolerance'])

### Module B answers

- **2D plots mislead:** Jaccard overlap between 2D and 100D neighbors is often near zero—projection collapses geometry.
- **Rotation invariance:** Orthogonal rotation of embedding columns preserves cosine neighbors and PIP on normalized vectors (all queries unchanged in our check).
- **Reconstruction vs semantics:** Reconstruction error decreases monotonically with k, but neighbor quality (Jaccard vs k=100) plateaus slowly—lower error does not guarantee better semantic neighbors.

## Module D — Basic polysemy (GloVe full 400k)

In [ ]:
poly = load_json(Path(runs['module_d']) / 'module_d_polysemy.json')
for word, info in poly.items():
    print(f"\n=== {word} (IMDB count={info['in_imdb_subset_count']}) ===")
    print('top-10:', [w for w, s in info['top10']])
    print('annotated:', info['top10_annotated'])
    if info.get('data_limit'):
        print('DATA LIMIT:', info['data_limit'])

### Module D answers

- Missing a sense in top-10 does **not** mean the vector lacks it—it may be averaged out or ranked below frequency-dominated neighbors.
- Frequent senses dominate the single vector (e.g. *bank* financial vs river in GloVe top-10).
- Neighbor clusters approximate senses but are not guaranteed to align with dictionary entries.

## Module E — Analogy (full GloVe)

In [ ]:
analogy = load_json(Path(runs['module_e']) / 'module_e_analogy.json')
summary = analogy['summary']
rows = []
for key, s in summary.items():
    rows.append({
        'relation_offset': key,
        'top1': s['top1_accuracy'],
        'top5': s['top5_accuracy'],
        'n_evaluable': s['n_evaluable'],
        'n_oov': s['n_oov'],
    })
pd.DataFrame(rows)

In [ ]:
print('Known failure cases (not in accuracy denominator):')
for fc in analogy['failure_cases']:
    print(fc['name'], 'rank=', fc['rank'], 'top5=', fc['top5'])

### Module E answers

- Offset direction similarity does not guarantee retrieving the gold word (family relations, polysemy).
- Frequency and hub words dominate candidate rankings; excluding input words is necessary but not sufficient.
- Multiple consistent test pairs across relations are stronger evidence than one famous example.

Single vs average offset: family relation ranks differ slightly (grandmother rank 18 vs 11) but top-1/top-5 aggregate accuracy is identical for our fixed test set.

## Baseline archive

Exported from `f97ea793`, executed copy at `../results/baseline/exploring_word_vectors_executed.ipynb`. See `notebook_mismatches.json` for documented assignment inconsistencies.

## Placeholders — not run in Phase 1

### Module C
CBOW / SGNS on `train[11500:13500]`, window / subsampling / negative-sample ablations, multi-seed. Same-corpus GloVe training deferred to Phase 3.

### Module D advanced
Context-vector clustering; simplified sparse coding (not full Arora 2018 reproduction).

### Module F
Word similarity rank correlation, cross-task table, optional IMDB sentiment downstream. Bias items remain in the assignment notebook; evaluation-coverage note only.

## Paper notes

See `../papers/` for Mikolov 2013a/b, GloVe 2014, Levy 2015, Arora 2016, Schnabel 2015, Arora 2018, Yin & Shen 2018.